# ⚡ 07. Generation 4 Foundation Model: Amazon Chronos-T5 (Zero-Shot)

Zero-shot probabilistic inference directly on raw univariate time series.

In [ ]:
!pip install -q git+https://github.com/amazon-science/chronos-forecasting.git

import torch
import numpy as np
from chronos import ChronosPipeline

# Load pretrained Chronos small variant
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

def predict_chronos(context_series, horizon=24, num_samples=200):
    context = torch.tensor(context_series.values, dtype=torch.float32).unsqueeze(0)
    forecast = pipeline.predict(
        context,
        prediction_length=horizon,
        num_samples=num_samples,
        limit_prediction_length=False,
    )
    samples = forecast[0].cpu().numpy()
    q10 = np.quantile(samples, 0.10, axis=0)
    q50 = np.quantile(samples, 0.50, axis=0)
    q90 = np.quantile(samples, 0.90, axis=0)
    return q10, q50, q90
